# Preparación del dataset Chákṣu

Este notebook prepara el dataset Chákṣu previamente auditado para los
experimentos de clasificación binaria de riesgo de Glaucoma.

Objetivos:

1. Reconstruir la metadata consolidada de las 1,345 retinografías.
2. Preservar las etiquetas originales proporcionadas por Chákṣu.
3. Incorporar la nomenclatura funcional utilizada por el proyecto.
4. Conservar la partición oficial Train/Test del dataset.
5. Crear una partición de validación exclusivamente a partir del Train oficial.
6. Mantener el Test oficial completamente aislado para la evaluación final.
7. Generar archivos de metadata reproducibles para los posteriores
   experimentos con ResNet50V2 y EfficientNetV2-M.

En esta etapa no se realiza Data Augmentation ni transformación de las
imágenes originales.

## 2. Importaciones

Importación de librerías para manipulación tabular y numérica.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

## 3. Configuración del proyecto

Definición de las rutas del proyecto y creación de la carpeta de metadata para el dataset maestro.

In [2]:
PROJECT_ROOT = Path(r"C:\Users\SeuNg720\Documents\TP1-RetiScan")
CHAKSU_ROOT = (
    PROJECT_ROOT
    / "datasets"
    / "glaucoma"
    / "chaksu"
)
RAW_DIR = CHAKSU_ROOT / "raw"
METADATA_DIR = CHAKSU_ROOT / "metadata"
METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Proyecto:", PROJECT_ROOT)
print("Chákṣu:", CHAKSU_ROOT)
print("Metadata:", METADATA_DIR)
print("\nComprobaciones:")
print("PROJECT_ROOT existe:", PROJECT_ROOT.exists())
print("CHAKSU_ROOT existe:", CHAKSU_ROOT.exists())
print("RAW_DIR existe:", RAW_DIR.exists())

Proyecto: C:\Users\SeuNg720\Documents\TP1-RetiScan
Chákṣu: C:\Users\SeuNg720\Documents\TP1-RetiScan\datasets\glaucoma\chaksu
Metadata: C:\Users\SeuNg720\Documents\TP1-RetiScan\datasets\glaucoma\chaksu\metadata

Comprobaciones:
PROJECT_ROOT existe: True
CHAKSU_ROOT existe: True
RAW_DIR existe: True


## 4. Definir las rutas oficiales

Definición y comprobación de existencia de las rutas originales para imágenes y decisiones de Train y Test.

In [3]:
TRAIN_IMAGES_DIR = RAW_DIR / "Train" / "1.0_Original_Fundus_Images"
TEST_IMAGES_DIR = RAW_DIR / "Test" / "1.0_Original_Fundus_Images"
TRAIN_DECISIONS_DIR = RAW_DIR / "Train" / "6.0_Glaucoma_Decision"
TEST_DECISIONS_DIR = RAW_DIR / "Test" / "6.0_Glaucoma_Decision"

rutas = {
    "Train images": TRAIN_IMAGES_DIR,
    "Test images": TEST_IMAGES_DIR,
    "Train decisions": TRAIN_DECISIONS_DIR,
    "Test decisions": TEST_DECISIONS_DIR,
}
for nombre, ruta in rutas.items():
    print(f"{nombre}: {ruta.exists()}")

Train images: True
Test images: True
Train decisions: True
Test decisions: True


## 5. Obtener las imágenes físicas

Conteo de retinografías físicas en disco para verificar la precondición estructural (1,009 en Train y 336 en Test, total 1,345).

In [4]:
EXTENSIONES_IMAGEN = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff"
}

def obtener_imagenes(carpeta):
    return [
        archivo
        for archivo in carpeta.rglob("*" )
        if archivo.is_file()
        and archivo.suffix.lower() in EXTENSIONES_IMAGEN
    ]

imagenes_train = obtener_imagenes(
    TRAIN_IMAGES_DIR
)
imagenes_test = obtener_imagenes(
    TEST_IMAGES_DIR
)
print("Train:", len(imagenes_train))
print("Test:", len(imagenes_test))
print(
    "Total:",
    len(imagenes_train) + len(imagenes_test)
)

Train: 1009
Test: 336
Total: 1345


## 6. Cargar las decisiones de Train

Lectura de los CSV de decisiones de Train y resolución determinística de las 104 imágenes de Bosch mediante correspondencia física con disco.

In [5]:
train_bosch_global = pd.read_csv(
    TRAIN_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Bosch_majority.csv"
)
train_forus = pd.read_csv(
    TRAIN_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Forus_majority.csv"
)
train_remidio = pd.read_csv(
    TRAIN_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Remidio_majority.csv"
)

train_bosch_global["image_name"] = (
    train_bosch_global["Images"]
    .str.extract(
        r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
        expand=False
    )
)
train_bosch_global["image_name_norm"] = (
    train_bosch_global["image_name"]
    .str.lower()
)

nombres_train_norm = {
    archivo.name.lower()
    for archivo in imagenes_train
}

train_bosch = train_bosch_global[
    train_bosch_global[
        "image_name_norm"
    ].isin(nombres_train_norm)
].copy()
print("Bosch Train:", len(train_bosch))

Bosch Train: 104


## 7. Preparar Forus y Remidio Train

Definición de `preparar_tabla` para resolver el nombre físico real del archivo a partir de su stem (solventando la discrepancia entre la extensión nominal del CSV y el archivo físico en disco) y consolidación de Train (1,009 registros).

In [6]:
mapa_archivos_fisicos = {
    f.stem.lower(): f.name
    for f in imagenes_train + imagenes_test
}

def preparar_tabla(
    df,
    camera,
    official_split
):
    df = df.copy()
    extracted = (
        df["Images"]
        .str.extract(
            r"^(.+?\.(?:jpg|jpeg|png|tif|tiff))-",
            expand=False
        )
    )
    stem = extracted.str.replace(r"\.[^.]+$", "", regex=True).str.lower()
    df["image_name"] = stem.map(mapa_archivos_fisicos)
    df["image_name_norm"] = df["image_name"].str.lower()
    df["camera"] = camera
    df["official_split"] = official_split
    return df

train_bosch = preparar_tabla(
    train_bosch,
    "Bosch",
    "train"
)
train_forus = preparar_tabla(
    train_forus,
    "Forus",
    "train"
)
train_remidio = preparar_tabla(
    train_remidio,
    "Remidio",
    "train"
)

train_metadata = pd.concat(
    [
        train_bosch,
        train_forus,
        train_remidio
    ],
    ignore_index=True
)
print("Metadata Train:", len(train_metadata))

Metadata Train: 1009


## 8. Preparar Test

Carga de decisiones de Test, estandarización de esquema de columna a `Majority Decision` y selección de las 41 imágenes físicas de Bosch en Test para consolidar los 336 registros oficiales.

In [7]:
test_bosch_global = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Bosch_majority.csv"
)
test_forus = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Forus_majority.csv"
)
test_remidio = pd.read_csv(
    TEST_DECISIONS_DIR
    / "Glaucoma_Decision_Comparison_Remidio_majority.csv"
)

test_forus = test_forus.rename(
    columns={
        "Glaucoma Decision":
        "Majority Decision"
    }
)
test_remidio = test_remidio.rename(
    columns={
        "Glaucoma Decision":
        "Majority Decision"
    }
)

test_bosch_global = preparar_tabla(
    test_bosch_global,
    "Bosch",
    "test"
)

nombres_test_norm = {
    archivo.name.lower()
    for archivo in imagenes_test
}

test_bosch = test_bosch_global[
    test_bosch_global[
        "image_name_norm"
    ].isin(nombres_test_norm)
].copy()
print(
    "Bosch Test:",
    len(test_bosch)
)

test_forus = preparar_tabla(
    test_forus,
    "Forus",
    "test"
)
test_remidio = preparar_tabla(
    test_remidio,
    "Remidio",
    "test"
)

test_metadata = pd.concat(
    [
        test_bosch,
        test_forus,
        test_remidio
    ],
    ignore_index=True
)
print(
    "Metadata Test:",
    len(test_metadata)
)

Bosch Test: 41
Metadata Test: 336


## 9. Consolidar las 1,345 imágenes

Unión de los metadatos de Train y Test en una tabla unificada `master`.

In [8]:
master = pd.concat(
    [
        train_metadata,
        test_metadata
    ],
    ignore_index=True
)
print("Total master:", len(master))

Total master: 1345


## 10. Conservar únicamente la información necesaria

Construcción del identificador unívoco `image_key` (`<camera>::<image_name_norm>`), renombramiento a `original_label` y depuración de columnas innecesarias.

In [9]:
master["image_key"] = (
    master["camera"]
    .str.lower()
    + "::"
    + master["image_name_norm"]
)

master_clean = master[
    [
        "image_key",
        "image_name",
        "image_name_norm",
        "camera",
        "Majority Decision",
        "official_split"
    ]
].copy()

master_clean = master_clean.rename(
    columns={
        "Majority Decision":
        "original_label"
    }
)

display(
    master_clean.head(10)
)

,image_key,image_name,image_name_norm,camera,original_label,official_split
0,bosch::image101.jpg,Image101.JPG,image101.jpg,Bosch,GLAUCOMA SUSPECT,train
1,bosch::image102.jpg,Image102.JPG,image102.jpg,Bosch,NORMAL,train
2,bosch::image103.jpg,Image103.JPG,image103.jpg,Bosch,NORMAL,train
3,bosch::image104.jpg,Image104.JPG,image104.jpg,Bosch,NORMAL,train
4,bosch::image105.jpg,Image105.JPG,image105.jpg,Bosch,NORMAL,train
5,bosch::image106.jpg,Image106.JPG,image106.jpg,Bosch,NORMAL,train
6,bosch::image107.jpg,Image107.JPG,image107.jpg,Bosch,GLAUCOMA SUSPECT,train
7,bosch::image108.jpg,Image108.JPG,image108.jpg,Bosch,NORMAL,train
8,bosch::image109.jpg,Image109.JPG,image109.jpg,Bosch,NORMAL,train
9,bosch::image110.jpg,Image110.JPG,image110.jpg,Bosch,NORMAL,train


## 11. Añadir la etiqueta funcional del proyecto

Incorporación de la nomenclatura funcional (`LOW_RISK` / `HIGH_RISK`) y numérica binaria (`0` / `1`), preservando la etiqueta original sin sobreescrituras.

In [10]:
MAPEO_FUNCIONAL = {
    "NORMAL": "LOW_RISK",
    "GLAUCOMA SUSPECT": "HIGH_RISK"
}
master_clean["functional_label"] = (
    master_clean["original_label"]
    .map(MAPEO_FUNCIONAL)
)

MAPEO_NUMERICO = {
    "NORMAL": 0,
    "GLAUCOMA SUSPECT": 1
}
master_clean["label"] = (
    master_clean["original_label"]
    .map(MAPEO_NUMERICO)
)

## 12. Verificar que el mapeo no produjo nulos

Comprobación de que todas las categorías fueron mapeadas exhaustivamente sin generar valores nulos.

In [11]:
print(
    "functional_label nulos:",
    master_clean[
        "functional_label"
    ].isna().sum()
)
print(
    "label nulos:",
    master_clean[
        "label"
    ].isna().sum()
)

functional_label nulos: 0
label nulos: 0


## 13. Añadir la ruta relativa de la imagen

Generación de rutas relativas con formato POSIX (`/`) referenciadas desde la raíz del proyecto para asegurar portabilidad entre diferentes entornos y máquinas.

In [12]:
mapa_rutas = {}
for archivo in imagenes_train:
    mapa_rutas[
        ("train", archivo.name.lower())
    ] = archivo.relative_to(
        PROJECT_ROOT
    ).as_posix()
for archivo in imagenes_test:
    mapa_rutas[
        ("test", archivo.name.lower())
    ] = archivo.relative_to(
        PROJECT_ROOT
    ).as_posix()

master_clean["image_path"] = (
    master_clean.apply(
        lambda fila: mapa_rutas.get(
            (
                fila["official_split"],
                fila["image_name_norm"]
            )
        ),
        axis=1
    )
)

print(
    "Rutas sin resolver:",
    master_clean["image_path"]
    .isna()
    .sum()
)

Rutas sin resolver: 0


## 14. Orden final del CSV maestro

Estructuración de las 8 columnas limpias y trazables del dataset maestro consolidado.

In [13]:
columnas_finales = [
    "image_key",
    "image_name",
    "image_path",
    "camera",
    "original_label",
    "functional_label",
    "label",
    "official_split"
]
chaksu_master = master_clean[
    columnas_finales
].copy()

display(
    chaksu_master.head(10)
)

,image_key,image_name,image_path,camera,original_label,functional_label,label,official_split
0,bosch::image101.jpg,Image101.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,GLAUCOMA SUSPECT,HIGH_RISK,1,train
1,bosch::image102.jpg,Image102.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
2,bosch::image103.jpg,Image103.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
3,bosch::image104.jpg,Image104.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
4,bosch::image105.jpg,Image105.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
5,bosch::image106.jpg,Image106.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
6,bosch::image107.jpg,Image107.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,GLAUCOMA SUSPECT,HIGH_RISK,1,train
7,bosch::image108.jpg,Image108.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
8,bosch::image109.jpg,Image109.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train
9,bosch::image110.jpg,Image110.JPG,datasets/glaucoma/chaksu/raw/Train/1.0_Origina...,Bosch,NORMAL,LOW_RISK,0,train


## 15. Validaciones del maestro

Verificación de integridad global: 1,345 filas, 1,345 identificadores únicos, 0 duplicados y 0 valores nulos.

In [14]:
print(
    "Filas:",
    len(chaksu_master)
)
print(
    "Image keys únicas:",
    chaksu_master[
        "image_key"
    ].nunique()
)
print(
    "Duplicados:",
    chaksu_master[
        "image_key"
    ].duplicated().sum()
)
print(
    "Nulos totales:",
    chaksu_master
    .isna()
    .sum()
    .sum()
)

Filas: 1345
Image keys únicas: 1345
Duplicados: 0
Nulos totales: 0


## 16. Comprobar la distribución final

Validación de la distribución por split y clase para contrastar con las cifras oficiales de Train y Test.

In [15]:
resumen_master = (
    chaksu_master
    .groupby(
        [
            "official_split",
            "original_label"
        ]
    )
    .size()
    .reset_index(
        name="cantidad"
    )
)
display(resumen_master)

,official_split,original_label,cantidad
0,test,GLAUCOMA SUSPECT,51
1,test,NORMAL,285
2,train,GLAUCOMA SUSPECT,137
3,train,NORMAL,872


## 17. Guardar chaksu_master.csv

Exportación formal del dataset maestro a `datasets/glaucoma/chaksu/metadata/chaksu_master.csv` y comprobación de su persistencia en disco.

In [16]:
MASTER_PATH = (
    METADATA_DIR
    / "chaksu_master.csv"
)
chaksu_master.to_csv(
    MASTER_PATH,
    index=False
)
print(
    "Archivo guardado en:",
    MASTER_PATH
)
print(
    "¿Existe?:",
    MASTER_PATH.exists()
)

Archivo guardado en: C:\Users\SeuNg720\Documents\TP1-RetiScan\datasets\glaucoma\chaksu\metadata\chaksu_master.csv
¿Existe?: True


## 18. Separación del Train oficial y Test oficial

A partir del archivo maestro se separan los subconjuntos oficiales definidos por
Chákṣu.

El subconjunto Test oficial se mantiene completamente aislado y no participa
en la creación de la partición de validación ni en decisiones posteriores de
entrenamiento o ajuste de hiperparámetros.

In [17]:
train_oficial = chaksu_master[
    chaksu_master["official_split"] == "train"
].copy()
test_final = chaksu_master[
    chaksu_master["official_split"] == "test"
].copy()

print("Train oficial:", len(train_oficial))
print("Test oficial:", len(test_final))

Train oficial: 1009
Test oficial: 336


## 19. Revisar distribución por cámara y clase en Train

Inspección de las cantidades cruzadas de cámara y etiqueta en Train oficial para diseñar una estratificación balanceada.

In [18]:
distribucion_train_camara = (
    train_oficial
    .groupby(
        ["camera", "original_label"]
    )
    .size()
    .reset_index(name="cantidad")
)
display(distribucion_train_camara)

,camera,original_label,cantidad
0,Bosch,GLAUCOMA SUSPECT,19
1,Bosch,NORMAL,85
2,Forus,GLAUCOMA SUSPECT,16
3,Forus,NORMAL,79
4,Remidio,GLAUCOMA SUSPECT,102
5,Remidio,NORMAL,708


## 20. Crear el estrato combinado

Construcción de la columna `stratum` combinando dispositivo y clase (`<CAMERA>__<ORIGINAL_LABEL>`) para garantizar que la partición conserve la representatividad de cada sensor y clase clínica.

In [19]:
train_oficial["stratum"] = (
    train_oficial["camera"]
    .str.upper()
    + "__"
    + train_oficial["original_label"]
)

print(
    train_oficial["stratum"]
    .value_counts()
)

stratum
REMIDIO__NORMAL              708
REMIDIO__GLAUCOMA SUSPECT    102
BOSCH__NORMAL                 85
FORUS__NORMAL                 79
BOSCH__GLAUCOMA SUSPECT       19
FORUS__GLAUCOMA SUSPECT       16
Name: count, dtype: int64


## 21. Crear Train y Validation reproducibles (80% / 20%)

División estratificada por `stratum` utilizando una semilla fija (`random_state=42`) para garantizar total reproducibilidad.

In [20]:
try:
    from sklearn.model_selection import train_test_split
except (ImportError, Exception):
    # Implementación equivalente en NumPy del algoritmo StratifiedShuffleSplit
    # para entornos restringidos por directivas de seguridad de DLLs
    def train_test_split(df, test_size=0.20, random_state=42, stratify=None):
        rng = np.random.RandomState(random_state)
        n_samples = len(df)
        n_test = int(np.ceil(test_size * n_samples)) if isinstance(test_size, float) else test_size
        n_train = n_samples - n_test
        y = stratify if stratify is not None else df["stratum"].values
        classes, y_indices, class_counts = np.unique(y, return_inverse=True, return_counts=True)
        n_classes = len(classes)
        class_indices = np.split(np.argsort(y_indices, kind="stable"), np.cumsum(class_counts)[:-1])
        
        continuous = class_counts / class_counts.sum() * n_train
        floored = np.floor(continuous)
        need_to_add = int(n_train - floored.sum())
        if need_to_add > 0:
            remainder = continuous - floored
            values = np.sort(np.unique(remainder))[::-1]
            for value in values:
                (inds,) = np.where(remainder == value)
                if len(inds) > need_to_add:
                    inds = rng.choice(inds, size=need_to_add, replace=False)
                floored[inds] += 1
                need_to_add -= len(inds)
                if need_to_add == 0:
                    break
        n_i = floored.astype(int)
        
        class_counts_remaining = class_counts - n_i
        continuous_t = class_counts_remaining / class_counts_remaining.sum() * n_test
        floored_t = np.floor(continuous_t)
        need_to_add_t = int(n_test - floored_t.sum())
        if need_to_add_t > 0:
            remainder_t = continuous_t - floored_t
            values_t = np.sort(np.unique(remainder_t))[::-1]
            for value in values_t:
                (inds,) = np.where(remainder_t == value)
                if len(inds) > need_to_add_t:
                    inds = rng.choice(inds, size=need_to_add_t, replace=False)
                floored_t[inds] += 1
                need_to_add_t -= len(inds)
                if need_to_add_t == 0:
                    break
        t_i = floored_t.astype(int)
        
        train_idx = []
        test_idx = []
        for i in range(n_classes):
            permutation = rng.permutation(class_counts[i])
            perm_indices = class_indices[i].take(permutation, mode="clip")
            train_idx.extend(perm_indices[: n_i[i]])
            test_idx.extend(perm_indices[n_i[i] : n_i[i] + t_i[i]])
        return df.iloc[rng.permutation(train_idx)].copy(), df.iloc[rng.permutation(test_idx)].copy()

RANDOM_STATE = 42
VALIDATION_SIZE = 0.20

train_exp, val_exp = train_test_split(
    train_oficial,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_oficial["stratum"]
)

print("Train experimental:", len(train_exp))
print("Validation:", len(val_exp))
print("Test final:", len(test_final))

Train experimental: 807
Validation: 202
Test final: 336


## 22. Comprobar la distribución por clase

Resumen porcentual de clases clínicas en los subconjuntos experimental Train, Validation y Test.

In [21]:
def resumen_clases(df, nombre_split):
    resumen = (
        df["original_label"]
        .value_counts()
        .rename_axis("original_label")
        .reset_index(name="cantidad")
    )
    resumen.insert(
        0,
        "split",
        nombre_split
    )
    resumen["porcentaje"] = (
        resumen["cantidad"]
        / len(df)
        * 100
    ).round(2)
    return resumen

resumen_train_exp = resumen_clases(
    train_exp,
    "train"
)
resumen_val_exp = resumen_clases(
    val_exp,
    "validation"
)
resumen_test_final = resumen_clases(
    test_final,
    "test"
)
resumen_splits = pd.concat(
    [
        resumen_train_exp,
        resumen_val_exp,
        resumen_test_final
    ],
    ignore_index=True
)
display(resumen_splits)

,split,original_label,cantidad,porcentaje
0,train,NORMAL,697,86.37
1,train,GLAUCOMA SUSPECT,110,13.63
2,validation,NORMAL,175,86.63
3,validation,GLAUCOMA SUSPECT,27,13.37
4,test,NORMAL,285,84.82
5,test,GLAUCOMA SUSPECT,51,15.18


## 23. Verificar distribución por cámara y clase

Comprobación cruzada para verificar que la partición estratificada preservó la proporción original de cada dispositivo y clase.

In [22]:
def resumen_camara_clase(
    df,
    nombre_split
):
    resumen = (
        df
        .groupby(
            ["camera", "original_label"]
        )
        .size()
        .reset_index(name="cantidad")
    )
    resumen.insert(
        0,
        "split",
        nombre_split
    )
    return resumen

resumen_camara_splits = pd.concat(
    [
        resumen_camara_clase(
            train_exp,
            "train"
        ),
        resumen_camara_clase(
            val_exp,
            "validation"
        ),
        resumen_camara_clase(
            test_final,
            "test"
        )
    ],
    ignore_index=True
)
display(resumen_camara_splits)

,split,camera,original_label,cantidad
0,train,Bosch,GLAUCOMA SUSPECT,15
1,train,Bosch,NORMAL,68
2,train,Forus,GLAUCOMA SUSPECT,13
3,train,Forus,NORMAL,63
4,train,Remidio,GLAUCOMA SUSPECT,82
5,train,Remidio,NORMAL,566
6,validation,Bosch,GLAUCOMA SUSPECT,4
7,validation,Bosch,NORMAL,17
8,validation,Forus,GLAUCOMA SUSPECT,3
9,validation,Forus,NORMAL,16


## 24. Verificar ausencia de solapamiento entre particiones

Comprobación formal de intersección de claves para descartar contaminación entre particiones (*data leakage*).

In [23]:
train_keys = set(
    train_exp["image_key"]
)
val_keys = set(
    val_exp["image_key"]
)
test_keys = set(
    test_final["image_key"]
)

print(
    "Train y Validation repetidos:",
    len(train_keys & val_keys)
)
print(
    "Train y Test repetidos:",
    len(train_keys & test_keys)
)
print(
    "Validation y Test repetidos:",
    len(val_keys & test_keys)
)

Train y Validation repetidos: 0
Train y Test repetidos: 0
Validation y Test repetidos: 0


## 25. Añadir el split experimental

Asignación de la columna `experimental_split` (`train`, `validation`, `test`) y remoción de la columna auxiliar `stratum`.

In [24]:
train_exp = train_exp.copy()
val_exp = val_exp.copy()
test_final = test_final.copy()

train_exp["experimental_split"] = "train"
val_exp["experimental_split"] = "validation"
test_final["experimental_split"] = "test"

train_exp = train_exp.drop(
    columns=["stratum"],
    errors="ignore"
)
val_exp = val_exp.drop(
    columns=["stratum"],
    errors="ignore"
)
test_final = test_final.drop(
    columns=["stratum"],
    errors="ignore"
)

## 26. Construir la metadata experimental completa

Consolidación de las particiones experimentales en `chaksu_splits` (total 1,345 registros).

In [25]:
chaksu_splits = pd.concat(
    [
        train_exp,
        val_exp,
        test_final
    ],
    ignore_index=True
)

print("Total:", len(chaksu_splits))
print(
    chaksu_splits[
        "experimental_split"
    ].value_counts()
)

Total: 1345
experimental_split
train         807
test          336
validation    202
Name: count, dtype: int64


## 27. Validación final de los splits

Control de calidad exhaustivo de `chaksu_splits`: 1,345 registros, 1,345 claves únicas, 0 duplicados y 0 valores nulos.

In [26]:
print(
    "Image keys únicas:",
    chaksu_splits[
        "image_key"
    ].nunique()
)
print(
    "Duplicados:",
    chaksu_splits[
        "image_key"
    ].duplicated().sum()
)
print(
    "Nulos:",
    chaksu_splits
    .isna()
    .sum()
    .sum()
)

Image keys únicas: 1345
Duplicados: 0
Nulos: 0


## 28. Guardar los splits en archivos de metadata

Persistencia en disco del conjunto completo `chaksu_splits.csv` y de los subconjuntos individuales `chaksu_train.csv`, `chaksu_validation.csv` y `chaksu_test.csv` para alimentar los futuros pipelines de modelado.

In [27]:
SPLITS_PATH = (
    METADATA_DIR
    / "chaksu_splits.csv"
)
TRAIN_PATH = (
    METADATA_DIR
    / "chaksu_train.csv"
)
VAL_PATH = (
    METADATA_DIR
    / "chaksu_validation.csv"
)
TEST_PATH = (
    METADATA_DIR
    / "chaksu_test.csv"
)

chaksu_splits.to_csv(
    SPLITS_PATH,
    index=False
)
train_exp.to_csv(
    TRAIN_PATH,
    index=False
)
val_exp.to_csv(
    VAL_PATH,
    index=False
)
test_final.to_csv(
    TEST_PATH,
    index=False
)

for ruta in [
    SPLITS_PATH,
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH
]:
    print(
        ruta.name,
        "→",
        ruta.exists()
    )

chaksu_splits.csv → True
chaksu_train.csv → True
chaksu_validation.csv → True
chaksu_test.csv → True


# Conclusiones de la preparación del dataset

Se construyó una metadata consolidada para las 1,345 retinografías del
dataset Chákṣu, preservando las etiquetas originales y la partición oficial
Train/Test.

La etiqueta original NORMAL se asoció funcionalmente con LOW_RISK y
GLAUCOMA SUSPECT con HIGH_RISK, conservándose además una codificación
numérica binaria de 0 y 1 respectivamente.

El subconjunto Train oficial de 1,009 imágenes fue dividido de manera
reproducible en entrenamiento y validación mediante estratificación conjunta
por dispositivo de adquisición y etiqueta original.

La distribución experimental resultante fue:

- Train: 807 imágenes.
- Validation: 202 imágenes.
- Test: 336 imágenes.

El subconjunto Test oficial se mantuvo completamente aislado de la generación
de las particiones de entrenamiento y validación.

No se detectaron imágenes repetidas entre los tres subconjuntos experimentales,
ni identificadores duplicados o valores nulos en la metadata final.

Las particiones generadas se almacenaron en archivos CSV y permanecerán
congeladas para asegurar que los modelos ResNet50V2 y EfficientNetV2-M sean
evaluados bajo exactamente las mismas condiciones experimentales.